# SkinCAP proposition-segments training

Full Colab training notebook for `RAD-colab-segment-query`. It preserves the P1/P2 schedule while training token-level alignment queries and saving post-hoc proposition attention maps.

In [ ]:
from pathlib import Path
import json
import os
import shutil
import subprocess

from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/trisle0805/RAD-colab.git'
REPO_DIR = Path('/content/RAD-colab')
BRANCH = 'RAD-colab-segment-query'
SKINCAP_ROOT = Path('/content/drive/MyDrive/Cao học/Experiments/SkinCAP')
IMAGE_ROOT = SKINCAP_ROOT / 'skincap'
TRAIN_CSV = SKINCAP_ROOT / 'skincap_47_train.csv'
VAL_CSV = SKINCAP_ROOT / 'skincap_47_val.csv'
TEST_CSV = SKINCAP_ROOT / 'skincap_47_test.csv'
BERT_DIR = Path('/content/models/Bio_ClinicalBERT')
EXPERIMENT_NAME = 'skin-proposition-segments'
RUN_ID = 'SEG_s42'
RUN_ROOT = Path('/content/drive/MyDrive/Cao học/Experiments') / EXPERIMENT_NAME
RUN_ROOT.mkdir(parents=True, exist_ok=True)

if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
subprocess.run(['git', 'checkout', BRANCH], check=True)
subprocess.run(['pip', 'install', '-r', 'requirements.txt'], check=True)

if not BERT_DIR.exists():
    BERT_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        'git', 'clone', 'https://huggingface.co/emilyalsentzer/Bio_ClinicalBERT', str(BERT_DIR)
    ], check=True)

for required in (TRAIN_CSV, VAL_CSV, TEST_CSV, IMAGE_ROOT, BERT_DIR):
    if not required.exists():
        raise FileNotFoundError(required)
print({'run_root': str(RUN_ROOT), 'branch': BRANCH, 'run_id': RUN_ID})

In [ ]:
# P1/P2-compatible training schedule: 30 epochs, warmup 10, batch 4, accumulation 4, seed 42.
import yaml

with (REPO_DIR / 'configs/skin.yaml').open(encoding='utf-8') as handle:
    config = yaml.safe_load(handle)
config['batch_size'] = 4
config['test_batch_size'] = 4
config['grad_accumulation_steps'] = 4
config['schedular']['epochs'] = 30
config['schedular']['warmup_epochs'] = 10
CONFIG_PATH = Path('/content/skin_segments.yaml')
with CONFIG_PATH.open('w', encoding='utf-8') as handle:
    yaml.safe_dump(config, handle, sort_keys=False)

OUTPUT_DIR = RUN_ROOT / RUN_ID
command = [
    'python', 'main_proposition.py',
    '--config', str(CONFIG_PATH),
    '--output_dir', str(OUTPUT_DIR),
    '--run_id', RUN_ID,
    '--train_csv', str(TRAIN_CSV), '--val_csv', str(VAL_CSV), '--test_csv', str(TEST_CSV),
    '--image_root', str(IMAGE_ROOT), '--bert_model_name', str(BERT_DIR),
    '--query_level', 'proposition_segments', '--query_chunk_size', '2048',
    '--seed', '42', '--device', 'cuda',
]
print(' '.join(command))
# Stream training logs directly; segment mode intentionally has no frozen or detached text queries.
subprocess.run(command, cwd=REPO_DIR, check=True)

In [ ]:
# Compare common attention-map metrics for P1, P2, and this segment run.
import pandas as pd

comparison = {
    'P1': Path('/content/drive/MyDrive/Cao học/Experiments/skin-proposition-20ep/20261007_112004/checklist/test_proposition_evidence.npz'),
    'P2': Path('/content/drive/MyDrive/Cao học/Experiments/skin-proposition-P2-frozenq/20261008_121605/checklist/test_proposition_evidence.npz'),
    'Segments': OUTPUT_DIR / 'checklist' / 'test_segment_evidence.npz',
}
rows = {}
for label, evidence in comparison.items():
    if not evidence.exists():
        rows[label] = {'status': f'missing: {evidence}'}
        continue
    subprocess.run([
        'python', 'proposition/visualization/segment_map_analysis.py', '--evidence', str(evidence)
    ], cwd=REPO_DIR, check=True)
    rows[label] = json.loads((evidence.parent / 'segment_map_analysis.json').read_text(encoding='utf-8'))

pd.DataFrame(rows)